In [2]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/enrichment_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/gene_mapping_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/top_corr_module_fxns.R")

Here I perform enrichment analysis to find modules enriched for cell type markers. 

These modules will later be used to correlate with exon PSI to define cell type-specific exons.

In [1]:
n_workers <- 20

In [ ]:
mod_def <- "PosBC"

In [ ]:
max_set_size <- 200
min_set_size <- 3

## Prep marker genes

### Claude marker genes

In [3]:
set_source <- "Claude_marker_genes"

marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")

### Broad 

In [ ]:
library(msigdbr)

In [ ]:
gs_GOBP <- msigdbr(collection="C5", subcollection="GO:BP")

gs_CTS <- msigdbr(collection="C8")
gs_CTS <- gs_CTS[gs_CTS$gs_geoid == "GSE104276",]

gs_GOBP_list <- tapply(gs_GOBP$gene_symbol, INDEX=gs_GOBP$gs_name, FUN=list)
gs_CTS_list <- tapply(gs_CTS$gene_symbol, INDEX=gs_CTS$gs_name, FUN=list)

gs_list <- c(gs_GOBP_list, gs_CTS_list)

In [ ]:
set_sizes <- lengths(gs_list)
mask <- (set_sizes < max_set_size) & (set_sizes > min_set_size)
gs_list <- gs_list[mask]

print(length(gs_list))

In [ ]:
set_source <- paste0("MSigDB_", length(gs_list), "sets")

### MO marker genes

In [ ]:
load("/mnt/lareaulab/reliscu/data/gene_sets/Oldham/MO_sets.RData")

names(marker_genes_list) <- MO_legend$SetName

mask <- !MO_legend$Category %in% "Disease"
marker_genes_list <- MO_sets[mask]

In [ ]:
set_sizes <- lengths(marker_genes_list)
mask <- (set_sizes < max_set_size) & (set_sizes > min_set_size)
marker_genes_list <- marker_genes_list[mask]

print(length(marker_genes_list))

In [ ]:
set_source <- paste0("MO_", length(marker_genes_list), "sets")

## Get enrichments

In [37]:
network_dir <- "GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules"

In [ ]:
enrichments_df <- get_module_enrichments_par(network_dir, marker_genes_list, mod_def, n_workers)

In [39]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

In [40]:
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
Micro/PVM,9.079137e-24,4.398842e-19,plum1,Bicor-None_signum0.781_minSize10_merge_ME_0.95_22162
Astro,4.074082e-18,4.386429e-14,khaki3,Bicor-None_signum0.881_minSize3_merge_ME_0.95_22162
Oligo,5.203321e-18,5.042018e-14,darkgrey,Bicor-None_signum0.663_minSize8_merge_ME_0.95_22162
All GABAergic,1.161883e-17,1.023514e-13,steelblue,Bicor-None_signum0.881_minSize4_merge_ME_0.95_22162
VLMC,1.478217e-12,4.939285e-09,thistle3,Bicor-None_signum0.731_minSize10_merge_ME_0.95_22162
Sst,1.108788e-10,2.387589e-07,greenyellow,Bicor-None_signum0.953_minSize3_merge_ME_0.95_22162
Endo,4.700871e-10,9.110287e-07,seagreen4,Bicor-None_signum0.693_minSize6_merge_ME_0.95_22162
Vip,7.647361e-09,9.880391e-06,magenta1,Bicor-None_signum0.881_minSize3_merge_ME_0.95_22162
OPC,2.159819e-08,2.491505e-05,orangered4,Bicor-None_signum0.928_minSize3_merge_ME_0.95_22162


In [41]:
get_mod_vs_DE_genes(top_mods_df, marker_genes_list, mod_def)

,Cell_type,Qval,Module_genes,DE_genes,Intersection_genes,Module,Network,ME_path,kME_path
,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Micro/PVM,Micro/PVM,4.398842e-19,"LAPTM5, NCKAP1L, CSF1R, TBXAS1, SYK, ITGB2, HAVCR2, IRF8, APBB1IP, SASH3, BTK, DOCK2, TYROBP, C1QA, TLR6","CTSS, CX3CR1, ITGAM, P2RY12, TMEM119, TYROBP, AIF1, C1QA, C1QB, C1QC, CSF1R, HEXB, SALL1, SPI1, TREM2","CTSS,CX3CR1,ITGAM,P2RY12,TMEM119,TYROBP,AIF1,C1QA,C1QB,C1QC,CSF1R,SPI1,TREM2,SELPLG",plum1,Bicor-None_signum0.781_minSize10_merge_ME_0.95_22162,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules/Bicor-None_signum0.781_minSize10_merge_ME_0.95_22162/Module_eigengenes_10-12-54.csv,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules/Bicor-None_signum0.781_minSize10_merge_ME_0.95_22162/kME_table_10-12-54.csv
Astro,Astro,4.386429e-14,"MSI2, SPON1, PLPP3, METTL7A, BMPR1B, SLC39A12, S1PR1, RANBP3L, HSDL2, SASH1, GLI3, ERLIN2, SDC2, FERMT2, TMBIM6","ALDH1L1, AQP4, GFAP, SLC1A2, SLC1A3, ALDOC, FGFR3, GJA1, GLUL, NDRG2, S100B, SOX9, ACSL6, AGT, APOE","AQP4,SLC1A2,SLC1A3,FGFR3,GJA1,GLUL,SOX9,AGT,APOE",khaki3,Bicor-None_signum0.881_minSize3_merge_ME_0.95_22162,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules/Bicor-None_signum0.881_minSize3_merge_ME_0.95_22162/Module_eigengenes_09-28-47.csv,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules/Bicor-None_signum0.881_minSize3_merge_ME_0.95_22162/kME_table_09-28-47.csv
Oligo,Oligo,5.042018e-14,"MAG, MOG, CNP, ENPP2, FA2H, GJB1, NINJ2, TMEM125, CLDN11, GPR62, KLK6, GPR37, SLC44A1, DBNDD2, PLLP","MBP, MOG, PLP1, CLDN11, CNP, ERMN, MAG, MOBP, OPALIN, ENPP6, FA2H, KLK6, TF, UGT8A, NA","MOG,PLP1,CLDN11,CNP,ERMN,MAG,OPALIN,FA2H,KLK6,TF",darkgrey,Bicor-None_signum0.663_minSize8_merge_ME_0.95_22162,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules/Bicor-None_signum0.663_minSize8_merge_ME_0.95_22162/Module_eigengenes_12-54-38.csv,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules/Bicor-None_signum0.663_minSize8_merge_ME_0.95_22162/kME_table_12-54-38.csv
All GABAergic,All GABAergic,1.023514e-13,"GAD2, ZNF385D, GAD1, NXPH2, VWC2, BTBD11, ARL4C, MYO5B, GJD2, LGI2, KCNS3, NXPH1, KCNAB1, GRK3, GRIP1","GAD1, GAD2, SLC32A1, DLX1, DLX2, DLX5, DLX6, NA, NA, NA, NA, NA, NA, NA, NA","GAD1,GAD2,SLC32A1,DLX1,DLX5,DLX6",steelblue,Bicor-None_signum0.881_minSize4_merge_ME_0.95_22162,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules/Bicor-None_signum0.881_minSize4_merge_ME_0.95_22162/Module_eigengenes_09-43-29.csv,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules/Bicor-None_signum0.881_minSize4_merge_ME_0.95_22162/kME_table_09-43-29.csv
VLMC,VLMC,4.939285e-09,"SLC13A4, FAM20A, DSP, STRA6, SLC5A5, SLC6A20, SERPIND1, PTGDR, INSRR, FMOD, MLPH, WNK4, SLC9A2, DSG2, BNC2","COL1A1, COL1A2, DCN, LUM, CD44, COL3A1, IL33, OGN, PTGDS, SLC6A13, SPP1, NA, NA, NA, NA","COL1A1,COL1A2,DCN,COL3A1,OGN,PTGDS,SLC6A13",thistle3,Bicor-None_signum0.731_minSize10_merge_ME_0.95_22162,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_co

In [42]:
write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments.csv"), row.names=FALSE, quote=FALSE)

## Compare enrichments

In [ ]:
# x <- list.files("data/enrichments")[grep(set_source, list.files("data/enrichments"))]

# enrichments_list <- lapply(x, function(y) {
#     split <- unlist(strsplit(y, "STAR_"))
#     prefix <- unlist(strsplit(split[1], "_ComBat"))[1]
#     middle <- gsub("p_", "p", unlist(strsplit(split[1], "yao_2021_"))[2])
#     suffix <- gsub("_enrichments.csv", "", unlist(strsplit(split[2], "DE_genes_"))[2])
#     dataname <- paste(prefix, paste(middle, suffix, sep="_"), sep="_")

#     enrich_df <- fread(file.path("data/enrichments", y), data.table=FALSE)
#     enrich_df <- enrich_df[,c("Cell_type", "Qval")]
#     colnames(enrich_df)[2] <- dataname
#     enrich_df
# })

# enrichment_table <- Reduce(f=function(x, y) {
#     merge(x, y, by="Cell_type", all=TRUE)
# }, enrichments_list)

In [ ]:
# enrichment_table

Cell_type,GTEx_cortex_counts_All_501_outliers_removed_NA_NA,GTEx_cortex_TPM_All_598_outliers_removed_NA_NA,GTEx_cortex_TPM_QN_rd2_All_501_outliers_removed_NA_NA
<chr>,<dbl>,<dbl>,<dbl>
All GABAergic,1.023514e-13,1.351891e-14,3.077119e-12
All Glutamatergic,2.466413e-03,1.473401e-02,2.590165e-02
All Neuronal,1.245101e-04,5.003005e-04,6.197873e-02
Astro,4.386429e-14,2.015052e-15,9.045933e-09
CGE Class,1.998021e-04,1.033153e-04,1.103737e-04
Chandelier,3.399612e-01,1.106190e-01,1.603232e-01
Endo,9.110287e-07,1.827541e-04,4.395923e-07
L2/3 IT,5.149739e-04,1.600087e-06,1.195509e-05
L4 IT,2.537585e-02,1.028264e-03,4.809219e-01


In [ ]:
# data.frame(colMeans(enrichment_table[,-1], na.rm=TRUE))

,colMeans.enrichment_table....1...na.rm...TRUE.
,<dbl>
GTEx_cortex_counts_All_501_outliers_removed_NA_NA,0.041606701
GTEx_cortex_TPM_All_598_outliers_removed_NA_NA,0.008475941
GTEx_cortex_TPM_QN_rd2_All_501_outliers_removed_NA_NA,0.083171611
